# Browser-Use Download Functionality Tutorial

## Overview

This tutorial demonstrates how to enable file download capabilities in Browser-Use for remote browser environments. The download patch allows agents to download files using HTTP requests instead of relying on browser file dialogs, which don't work in headless/remote scenarios.

### Key Features

* **Remote Browser Downloads**: Download files in headless/remote browser environments
* **Agent Download Awareness**: LLM sees download progress and failures in real-time
* **HTTP Client Fallback**: Uses HTTP requests when browser dialogs fail
* **Progress Tracking**: Monitor download progress and handle failures gracefully
* **Backward Compatible**: Works with existing Browser-Use code

## Prerequisites

Before running this tutorial, ensure you have:

* Python 3.11+
* Valid AWS credentials configured
* **`download-patch.zip`** file in the same directory as this notebook

**Important**: The `download-patch.zip` file contains the enhanced browser-use files and must be present in the same folder as this notebook for the patch to work.

## 1. Installation and Setup

### 1.1 Install Dependencies

In [ ]:
# Install specific browser-use version for patch compatibility
!pip install browser-use==0.11.2 bedrock-agentcore boto3 rich --quiet

### 1.2 Apply Download Functionality Patch

In [ ]:
%%writefile apply_download_patch.py
#!/usr/bin/env python3
"""
Browser-Use Download Functionality Patch
Applies download enhancements to browser-use 0.11.2 installation
"""

import os
import sys
import tempfile
import zipfile
import shutil
import subprocess
from pathlib import Path

def find_browser_use_installation():
    """Find the browser-use installation path"""
    try:
        result = subprocess.run([sys.executable, "-c", "import browser_use; print(browser_use.__file__)"], 
                              capture_output=True, text=True, check=True)
        browser_use_init = result.stdout.strip()
        return Path(browser_use_init).parent
    except subprocess.CalledProcessError:
        print("❌ Error: browser-use not found. Please install browser-use==0.11.2 first")
        sys.exit(1)

def apply_patch():
    """Apply the download functionality patch"""
    print("🔧 Applying browser-use download functionality patch...")
    
    # Find installation path
    install_path = find_browser_use_installation()
    print(f"📍 Found browser-use installation: {install_path}")
    
    # Get script directory and zip file
    script_dir = Path(__file__).parent
    zip_file = script_dir / "download-patch.zip"
    
    if not zip_file.exists():
        print(f"❌ Error: {zip_file} not found")
        sys.exit(1)
    
    # Create temp directory and extract
    with tempfile.TemporaryDirectory() as temp_dir:
        print("📦 Extracting patch files...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
        
        patch_dir = Path(temp_dir) / "download-patch"
        
        # File mapping: source -> destination
        file_mapping = {
            "downloads_watchdog.py": "browser/watchdogs/downloads_watchdog.py",
            "download_manager.py": "browser/download_manager.py", 
            "session.py": "browser/session.py",
            "profile.py": "browser/profile.py",
            "message_manager_service.py": "agent/message_manager/service.py",
            "prompts.py": "agent/prompts.py",
            "agent_service.py": "agent/service.py"
        }
        
        # Copy files
        for source_file, dest_path in file_mapping.items():
            source = patch_dir / source_file
            destination = install_path / dest_path
            
            if not source.exists():
                print(f"❌ Warning: {source_file} not found in patch")
                continue
                
            print(f"📄 Copying {source_file}")
            print(f"   From: {source}")
            print(f"   To:   {destination}")
            shutil.copy2(source, destination)
    
    print("✅ Download functionality patch applied successfully!")
    print("\n🎯 Usage:")
    print("  Set download_from_remote_browser=True in your browser profile")
    print("  Downloads will use HTTP client instead of browser file dialogs")

if __name__ == "__main__":
    apply_patch()

In [ ]:
# Apply the download functionality patch
!python apply_download_patch.py

## 2. Download Example with BrowserClient

### 2.1 Large File Download Example

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use.llm import ChatAnthropicBedrock
from browser_use import Agent
from browser_use import Browser, BrowserProfile
from rich.console import Console
from contextlib import suppress
import asyncio
from boto3.session import Session

console = Console()

async def download_large_file_example():
    """Example of downloading a large file with enhanced browser-use"""
    
    boto_session = Session()
    region = boto_session.region_name
    
    client = BrowserClient(region)
    client.start(viewport={'width': 1920, 'height': 1080})

    ws_url, headers = client.generate_ws_headers()
    browser_session = None

    try:
        # Create browser profile with download enhancement enabled
        browser_profile = BrowserProfile(
            headers=headers,
            timeout=1500000,
            downloads_path="./downloads",
            download_from_remote_browser=True,  # Enable HTTP download fallback
            auto_download_pdfs=True
        )

        browser_session = Browser(
            cdp_url=ws_url,
            browser_profile=browser_profile,
            keep_alive=True
        )
        
        await browser_session.start()
        
        bedrock_chat = ChatAnthropicBedrock(
            model='us.anthropic.claude-3-7-sonnet-20250219-v1:0',
            aws_region='us-west-2'
        )

        task = "Go to https://proof.ovh.net/files and download the 100 MB file. Please wait for the download to finish and tell me when done."

        agent = Agent(
            task=task,
            llm=bedrock_chat,
            browser_session=browser_session,
            llm_timeout=300
        )
        
        result = await agent.run()
        print(f"Download task completed: {result}")
        
        return result

    finally:
        if browser_session:
            with suppress(Exception):
                await browser_session.close()
        client.stop()

# Run the example
await download_large_file_example()

### 2.2 Verify Download Functionality

In [ ]:
# Check if download patch is properly applied
def verify_download_patch():
    """Verify that the download functionality is properly installed"""
    try:
        from browser_use.browser.download_manager import DownloadManager
        from browser_use import BrowserProfile
        
        # Check if download_from_remote_browser parameter exists
        profile = BrowserProfile(download_from_remote_browser=True)
        
        print("✅ Download functionality patch is properly installed")
        print(f"✅ download_from_remote_browser setting: {profile.download_from_remote_browser}")
        return True
        
    except ImportError as e:
        print(f"❌ Download patch not properly installed: {e}")
        return False
    except Exception as e:
        print(f"❌ Error verifying patch: {e}")
        return False

# Verify installation
verify_download_patch()

### 2.3 Check Downloaded Files

In [ ]:
import os
from pathlib import Path

# Check what files were downloaded
downloads_dir = Path("./downloads")
if downloads_dir.exists():
    downloaded_files = list(downloads_dir.glob("*"))
    print(f"📁 Found {len(downloaded_files)} downloaded files:")
    for file in downloaded_files:
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"  - {file.name} ({size_mb:.2f} MB)")
else:
    print("📁 Downloads directory not found")

## 3. Configuration Options

The download enhancement adds the following configuration option to `BrowserProfile`:

- **`download_from_remote_browser`** (bool, default: False)
  - `True`: Use HTTP client for downloads (recommended for remote/headless browsers)
  - `False`: Use standard browser download behavior (works for local browsers with file dialogs)

### Best Practices

1. **Always set `download_from_remote_browser=True`** for headless or remote browser scenarios
2. **Specify a `downloads_path`** to control where files are saved
3. **Monitor agent context** - the LLM will see download progress and can make intelligent decisions
4. **Handle failures gracefully** - the agent can retry with different approaches if downloads fail

### Agent Download Awareness

The enhanced agent will see download status in its context:
```
<downloads_in_progress>
Downloading: 100Mio.dat (45%, 12s elapsed)
These downloads are still in progress
</downloads_in_progress>

<failed_downloads>
Failed: file.pdf (Network error) 3m ago
These downloads failed recently
</failed_downloads>
```

## Conclusion

This tutorial demonstrated how to enhance Browser-Use with download functionality for remote browser environments. Key takeaways:

- **Download patch enables file downloads** in headless/remote browser scenarios
- **Agent awareness** allows the LLM to monitor download progress and handle failures
- **HTTP client fallback** works when browser file dialogs are not available
- **BrowserClient integration** works seamlessly with bedrock-agentcore

The enhanced download functionality is particularly useful for:
- Document processing workflows
- Data collection tasks
- Remote browser automation
- Large file downloads with progress monitoring